In [4]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')


In [5]:
df = pd.read_csv('data/near_earth_asteroids_2025 (1).csv')  
df.shape

C:\Users\Thisali Aturusinghe\AppData\Local\Temp\ipykernel_25556\2402189128.py:1: DtypeWarning: Columns (3: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/near_earth_asteroids_2025 (1).csv')


(41281, 29)

In [6]:
df.head()

,spkid,full_name,pdes,name,pha,H,diameter_km,diameter_m,diameter_is_estimated,size_category,albedo,rot_per,class,e,a,i,q,ad,per,per_y,moid_au,moid_km,moid_lunar_distances,n,condition_code,first_obs,last_obs,data_arc,data_arc_years
0,20000433,433 Eros (A898 PA),433,Eros,False,10.39,16.84000,16840.0,False,Large (>1 km) — City killer+,0.250,5.270,AMO,0.2228,1.458,10.83,1.133,1.78,643.0,1.76,0.1480,22140485.0,57.60,0.5598,0.0,1893-10-29,2021-05-13,46582.0,127.53
1,20000719,719 Albert (A911 TB),719,Albert,False,15.59,2.70683,2706.8,True,Large (>1 km) — City killer+,NaN,5.801,AMO,0.5466,2.637,11.57,1.195,4.08,1560.0,4.28,0.2010,30069172.0,78.22,0.2302,0.0,1911-10-04,2026-03-20,41806.0,114.46
2,20000887,887 Alinda (A918 AA),887,Alinda,False,13.82,4.20000,4200.0,False,Large (>1 km) — City killer+,0.310,28.410,AMO,0.5712,2.474,9.40,1.061,3.89,1420.0,3.89,0.0797,11922950.0,31.02,0.2533,0.0,1918-02-09,2025-08-27,39281.0,107.55
3,20001036,1036 Ganymed (A924 UB),1036,Ganymed,False,9.17,37.67500,37675.0,False,Large (>1 km) — City killer+,0.238,10.297,AMO,0.5332,2.665,26.68,1.244,4.09,1590.0,4.35,0.3430,51312070.0,133.49,0.2266,0.0,1924-10-23,2026-03-22,37040.0,101.41
4,20001221,1221 Amor (1932 EA1),1221,Amor,False,17.37,1.00000,1000.0,False,Large (>1 km) — City killer+,NaN,NaN,AMO,0.4346,1.920,11.87,1.085,2.75,972.0,2.66,0.1080,16156570.0,42.03,0.3705,0.0,1932-03-12,2025-02-19,33947.0,92.94


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41281 entries, 0 to 41280
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   spkid                  41281 non-null  int64  
 1   full_name              41281 non-null  str    
 2   pdes                   41281 non-null  str    
 3   name                   182 non-null    str    
 4   pha                    41281 non-null  bool   
 5   H                      41279 non-null  float64
 6   diameter_km            41281 non-null  float64
 7   diameter_m             41281 non-null  float64
 8   diameter_is_estimated  41281 non-null  bool   
 9   size_category          41281 non-null  str    
 10  albedo                 1204 non-null   float64
 11  rot_per                2181 non-null   float64
 12  class                  41281 non-null  str    
 13  e                      41281 non-null  float64
 14  a                      41281 non-null  float64
 15  i            

In [8]:
#dropping unnecessary columns
df = df.drop(columns=["moid_au", "moid_km","moid_lunar_distances","diameter_km","diameter_m","diameter_is_estimated", "spkid", "full_name", "pdes","name"])

In [9]:
df.isnull().sum()

pha                   0
H                     2
size_category         0
albedo            40077
rot_per           39100
class                 0
e                     0
a                     0
i                     0
q                     0
ad                    0
per                   0
per_y                 0
n                     0
condition_code        2
first_obs             1
last_obs              0
data_arc            406
data_arc_years      406
dtype: int64

In [10]:
#dropping cols with too many missing values
df = df.drop(columns=["albedo", "rot_per"])

In [11]:
#using median values to fill missing values
imputer = SimpleImputer(strategy="median")

df[["data_arc", "data_arc_years"]] = imputer.fit_transform(df[["data_arc", "data_arc_years"]])

In [12]:
#dropping rows with missing cols
df = df.dropna(subset=["H", "condition_code","first_obs"])

Converting bool values into binary (next 3 code blocks)

In [13]:
df["pha_binary"] = df["pha"].map({True: 1, False: 0})

In [14]:
df = df.drop(columns=["pha"])

In [15]:
df = df.rename(columns={"pha_binary" : "pha"})

In [16]:
#dropping unnecessary cols
df = df.drop(columns=["first_obs", "last_obs"])

In [17]:
#doing one hot encoding (This code block and next)
df = pd.get_dummies(df, columns=['class'], prefix='class')

In [18]:
class_cols = [c for c in df.columns if c.startswith('class_')]
df[class_cols] = df[class_cols].astype(int)

In [19]:
#Scaling numeric featueres

num_features = ["H", "e","a","i","q","ad","per","per_y","n"]

scaler = StandardScaler()

scaledIntdf = df.copy()
scaledIntdf[num_features] = scaler.fit_transform(df[num_features])

print(scaledIntdf)

              H                        size_category         e         a  \
0     -4.633251         Large (>1 km) — City killer+ -1.195361 -0.147927   
1     -2.825613         Large (>1 km) — City killer+  0.634539  0.441407   
2     -3.440905         Large (>1 km) — City killer+  0.773562  0.359930   
3     -5.057350         Large (>1 km) — City killer+  0.558812  0.455403   
4     -2.206845         Large (>1 km) — City killer+  0.001590  0.083008   
...         ...                                  ...       ...       ...   
41276 -0.997118  Medium (140m-1km) — Regional damage  0.409051  0.281452   
41277  1.331953      Tiny (<25m) — Airburst/harmless  0.748696  0.181480   
41278  1.165094      Tiny (<25m) — Airburst/harmless  1.706032  0.466400   
41279  0.713185      Tiny (<25m) — Airburst/harmless  0.467260  0.134493   
41280 -1.111834  Medium (140m-1km) — Regional damage  1.285007  0.533381   

              i         q        ad       per     per_y         n  \
0     -0.095028  0

In [20]:
#normalize values 
df['pha'].value_counts(normalize=True)

pha
0    0.938487
1    0.061513
Name: proportion, dtype: float64

In [21]:
X = df.drop(columns=['pha'])  
y = df['pha']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,        
    random_state=42
)

In [23]:
numeric_cols = ['H', 'e', 'a', 'i', 'q', 'ad', 'per', 'per_y', 'n',
                 'condition_code', 'data_arc', 'data_arc_years']

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

joblib.dump(scaler, 'data/scaler.pkl')  # re-save with the corrected fit

['data/scaler.pkl']

In [24]:
X_train.dtypes.value_counts()

float64    12
int64       4
str         1
Name: count, dtype: int64

In [29]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

In [30]:
'size_category' in X_train.columns

False

In [31]:
X_train.select_dtypes(include=['object', 'str']).columns

Index([], dtype='str')

In [33]:
X_train = X_train.drop(columns=['size_category'], errors='ignore')
X_test = X_test.drop(columns=['size_category'], errors='ignore')

In [34]:
#Saving everything

joblib.dump(X_train, 'data/X_train.pkl')
joblib.dump(X_test, 'data/X_test.pkl')
joblib.dump(y_train, 'data/y_train.pkl')
joblib.dump(y_test, 'data/y_test.pkl')

joblib.dump(X_train_resampled, 'data/X_train_resampled.pkl')
joblib.dump(y_train_resampled, 'data/y_train_resampled.pkl')

['data/y_train_resampled.pkl']

In [35]:
joblib.dump(scaler, 'data/scaler.pkl')

['data/scaler.pkl']